In [1]:
import warnings
warnings.filterwarnings("ignore")
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
import numpy as np



Instructions for updating:
non-resource variables are not supported in the long term


단어 품사 구분하기

'I work at google', 'I google at work' 문장의 품사를 구분하는 RNN 코드를 만든다.

각 단어는 원-핫 인코딩으로 표현한다.  
I&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;=> `[1, 0, 0, 0]` # 대명사  
work&nbsp;&nbsp;&nbsp;=> `[0, 1, 0, 0]` # 동사  
at&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;=> `[0, 0, 1, 0]` # 전치사  
google => `[0, 0, 0, 1]` # 명사

I work at google => `[[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]]`  
I google at work => `[[1, 0, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0], [0, 1, 0, 0]]`

In [2]:
inputs = np.array([
    [[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]], # I work at google
    [[1, 0, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0], [0, 1, 0, 0]]  # I google at work
])
inputs

array([[[1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 1, 0],
        [0, 0, 0, 1]],

       [[1, 0, 0, 0],
        [0, 0, 0, 1],
        [0, 0, 1, 0],
        [0, 1, 0, 0]]])

In [3]:
rnn_inputs = tf.constant(inputs, dtype=tf.float32)
rnn_cell = tf.nn.rnn_cell.BasicRNNCell(num_units=3)
outputs, state = tf.nn.dynamic_rnn(cell=rnn_cell, inputs=rnn_inputs, dtype=tf.float32)
print('=' * 100)

print('출력값: {}'.format(outputs))
print('상태값: {}'.format(state))
print('=' * 100)

print('가중치의 개수와 바이어스의 개수')
for v in tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES):
    print(v)


Instructions for updating:
Please use `keras.layers.RNN(cell)`, which is equivalent to this API
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
출력값: Tensor("rnn/transpose_1:0", shape=(2, 4, 3), dtype=float32)
상태값: Tensor("rnn/while/Exit_3:0", shape=(2, 3), dtype=float32)
가중치의 개수와 바이어스의 개수
<tf.Variable 'rnn/basic_rnn_cell/kernel:0' shape=(7, 3) dtype=float32_ref>
<tf.Variable 'rnn/basic_rnn_cell/bias:0' shape=(3,) dtype=float32_ref>


In [32]:
var_name = [v.name for v in tf.trainable_variables()]
with tf.Session() as sess:
    sess.run(tf.global_variables_initializer())
    _outputs, _state = sess.run([outputs, state])
    
    # 두 문장의 각 RNN 셀의 첫 단어의 출력값이 같다. 첫 단어의 출력값이 같은 이유는 이전 단어가 없기 때문에 이전 상태값이 없기 때문이다.
    # 두 문장의 각 RNN 셀의 두 번째 단어부터는 출력값이 다르다. 이전 단어의 상태값이 현재 단어의 출력에 영향을 미치기 때문이다.
    print('출력값\n', _outputs, sep='')
    # 첫 번째 문장의 마지막 출력값은 최종 상태값과 같고 두 번째 문장의 마지막 출력값 역시 최종 상태값과 같다.
    print('최종 상태값\n', _state, sep='')
    print('=' * 100)
    
    values = sess.run(var_name)
    print('가중치\n', var_name[0], '\n', values[0], sep='')
    print('바이어스\n', var_name[1], '\n', values[1], sep='')

출력값
[[[ 0.51468766 -0.10224466  0.37632787]
  [ 0.20840079 -0.33679593  0.10422298]
  [-0.6323127  -0.6491268   0.51829636]
  [ 0.15640688 -0.00775192  0.07436518]]

 [[ 0.51468766 -0.10224466  0.37632787]
  [ 0.36389884  0.24604535  0.12892966]
  [-0.5086195  -0.4404695   0.23082405]
  [ 0.11297738 -0.58078206  0.09544594]]]
최종 상태값
[[ 0.15640688 -0.00775192  0.07436518]
 [ 0.11297738 -0.58078206  0.09544594]]
가중치
rnn/basic_rnn_cell/kernel:0
[[ 0.56908596 -0.1026032   0.39577472]
 [ 0.3090527  -0.5494025   0.07768601]
 [-0.62075216 -0.68721783  0.35065067]
 [ 0.4789263   0.05227059  0.10273445]
 [ 0.01808995  0.06936997  0.29045248]
 [ 0.31989837  0.4740399  -0.63788104]
 [-0.19705296  0.5625216  -0.49902076]]
바이어스
rnn/basic_rnn_cell/bias:0
[0. 0. 0.]
